# Exercise 6 — A Small Nonlinear Solver

**Companion to Lecture 6** (Nonlinear Problems and Newton's Method) of the lecture notes
*Essential Concepts in FEM and Physics-Based ML*.

You build, step by step, the solver that Lecture 6 describes: a Newton iteration, an assembly of
internal forces and tangent stiffness for springs with arbitrary laws, load stepping, displacement
control, and contact with a Lagrange multiplier. Along the way: the most useful debugging habit in
nonlinear FEM (checking the tangent), and what a wrong tangent does.

Only `numpy` and `matplotlib`. Fill in every `...`; each part ends with a ✅ check cell.

---
*Author: Ramses Sala · Copyright © 2025 Ramses Sala · Part of the course* Essential Concepts in FEM and Physics-Based ML. *Created with assistance of Claude Code for text, image and code refinement and generation.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=5, suppress=True)


---
## Part 1 — Newton's method for one equation

Write `newton_scalar(r, dr, u0)`: repeat $u \leftarrow u - r(u)/r'(u)$ until $|r(u)| <$ `tol`, and
return the list of all iterates.

1. Test it on $u + u^3 = 2$ from $u_0 = 0$ (lecture table: 0, 2, 1.3846, 1.0826, 1.0048, …).
2. Compute $\sqrt 2$ with it, using nothing but multiplication and division.
3. Quadratic convergence means $e_{k+1} \approx C\,e_k^2$. Print $e_{k+1}/e_k^2$ for the first problem:
   does it settle? (Theory: $C = |r''/(2r')|$ at the solution $= 6/8$.)

In [ ]:
def newton_scalar(r, dr, u0, tol=1e-12, maxit=50):
    iterates = [u0]
    u = u0
    for k in range(maxit):
        if abs(r(u)) < tol:
            break
        u = ...                               # TODO: Newton update
        iterates.append(u)
    return iterates

its = newton_scalar(lambda u: u + u**3 - 2, lambda u: 1 + 3*u**2, 0.0)
print("iterates:", np.round(its, 5))
sq = newton_scalar(..., ..., 1.0)             # TODO: r(x) = x^2 - 2
print("sqrt(2):", sq[-1], " iterates:", [f"{x:.10f}" for x in sq])
e = [abs(x - 1.0) for x in its]
print("e_{k+1}/e_k^2:", [round(e[k+1]/e[k]**2, 3) for k in range(2, 5)])

In [ ]:
# ✅ check — Part 1
assert np.allclose(its[:5], [0, 2, 1.384615, 1.082594, 1.004779], atol=1e-5), "Compare with the table in Section 6.3."
assert abs(sq[-1] - np.sqrt(2)) < 1e-12 and np.allclose(sq[:3], [1, 1.5, 1.4166667])
print("OK. The ratios approach 0.75: the number of correct digits doubles per step.")

---
## Part 2 — Internal forces and tangent stiffness for a chain

From now on a structure is a list of elements `(i, j, law)`: a spring between nodes `i` and `j`
whose **material law** `law(Δ)` returns the force $N$ and the tangent $k_t = dN/d\Delta$ for the
elongation $\Delta = u_j - u_i$. Three laws are given.

Write `assemble(u, elements)` returning the internal force vector $\mathbf f_{\rm int}$ and the tangent
matrix $\mathbf K_T$ for **all** nodes (Lecture 6, Section 6.5):
each element adds $-N$ to node $i$, $+N$ to node $j$, and $k_t\left[\begin{smallmatrix}1&-1\\-1&1\end{smallmatrix}\right]$ to $\mathbf K_T$.

In [ ]:
# given: three material laws, each returning (N, k_t)
linear    = lambda d: (d, 1.0)
stiffen   = lambda d: (d + d**3, 1 + 3*d**2)               # spring 1 of Lecture 6
soften    = lambda d: (d*np.exp(-d), (1 - d)*np.exp(-d))   # has a maximum force 1/e at d = 1

In [ ]:
def assemble(u, elements):
    n = len(u)
    f_int = np.zeros(n)
    K_T = np.zeros((n, n))
    for i, j, law in elements:
        N, kt = law(u[j] - u[i])
        ...                                   # TODO: add -N to node i and +N to node j
        ...                                   # TODO: add kt*[[1,-1],[-1,1]] to rows/cols (i, j)
    return f_int, K_T

# the three springs between two walls of Lecture 6: nodes 0..3, nodes 0 and 3 fixed
walls = [(0, 1, stiffen), (1, 2, linear), (2, 3, linear)]
f_test, K_test = assemble(np.array([0.0, 1.0, 0.5, 0.0]), walls)
f_test, K_test

In [ ]:
# ✅ check — Part 2
assert np.allclose(f_test, [-2.0, 2.5, 0.0, -0.5]), "At the solution of Lecture 6 the free nodes carry exactly the load (2.5, 0)."
assert np.allclose(K_test[1:3, 1:3], [[5, -1], [-1, 2]])
print("OK. Note f_int at the fixed nodes: these are the reactions (-2 and -0.5).")

---
## Part 3 — Newton for the whole structure

Write `newton_solve(u, f_ext, free, elements)`: the entries of `u` at non-free nodes are prescribed
(zero, or a given displacement) and stay fixed; Newton acts on the free entries only:
$$\mathbf r = (\mathbf f_{\rm int} - \mathbf f_{\rm ext})_{\rm free}, \qquad \mathbf K_{T,\rm ff}\,\Delta\mathbf u_{\rm free} = -\mathbf r .$$
Return the solution, the list of residual norms, and a flag `converged`. Stop when the norm is
below `tol`, or give up after `maxit` iterations (or if the numbers stop being finite).

Solve the wall example with $F = 2.5$ on node 1. Lecture: $\mathbf u = (1, 0.5)$, $N_1 = 2$.

In [ ]:
def newton_solve(u, f_ext, free, elements, tol=1e-10, maxit=50):
    u = u.astype(float).copy()
    norms = []
    for k in range(maxit):
        f_int, K_T = assemble(u, elements)
        r = ...                               # TODO: residual on the free entries
        norms.append(np.linalg.norm(r))
        if norms[-1] < tol:
            return u, norms, True
        du = ...                              # TODO: solve with the free-free block of K_T
        u[free] += du
        if not np.all(np.isfinite(u)):
            return u, norms, False
    return u, norms, False

free = [1, 2]
f_ext = np.array([0.0, 2.5, 0.0, 0.0])
u_sol, norms, ok = newton_solve(np.zeros(4), f_ext, free, walls)
print("u =", u_sol, " iterations:", len(norms) - 1, " residual norms:", [f"{x:.1e}" for x in norms])

In [ ]:
# ✅ check — Part 3
assert ok and np.allclose(u_sol[1:3], [1.0, 0.5], atol=1e-9)
assert len(norms) - 1 <= 7
print("OK. Six iterations from the unloaded state, the last ones each squaring the residual.")

---
## Part 4 — Check your tangent; then break it on purpose

The most common bug in nonlinear finite element code is a tangent that does not match the
internal forces. The cure is a habit: compare $\mathbf K_T$ with a **finite-difference** derivative,
column by column:
$$\mathbf K_T[:, j] \approx \frac{\mathbf f_{\rm int}(\mathbf u + \delta\,\mathbf e_j) - \mathbf f_{\rm int}(\mathbf u - \delta\,\mathbf e_j)}{2\delta}.$$

1. Implement the check at a random-looking state and print the largest difference.
2. Now introduce a typical bug: a law whose tangent forgot the factor 3, `(d + d**3, 1 + d**2)`.
   Does the finite-difference check catch it? Does Newton still find the right answer, and in how many iterations?
3. *Modified Newton* uses the **initial** tangent all the time. Try it with $F = 1$ and with $F = 2.5$.

In [ ]:
def fd_tangent(u, elements, delta=1e-6):
    n = len(u)
    K_fd = np.zeros((n, n))
    for j in range(n):
        e = np.zeros(n); e[j] = delta
        K_fd[:, j] = ...                      # TODO: central difference of f_int
    return K_fd

u_test = np.array([0.0, 0.7, 0.3, 0.0])
diff_ok = np.abs(assemble(u_test, walls)[1] - fd_tangent(u_test, walls)).max()

buggy = lambda d: (d + d**3, 1 + d**2)
walls_buggy = [(0, 1, buggy), (1, 2, linear), (2, 3, linear)]
diff_bug = np.abs(assemble(u_test, walls_buggy)[1] - fd_tangent(u_test, walls_buggy)).max()
u_bug, norms_bug, ok_bug = newton_solve(np.zeros(4), f_ext, free, walls_buggy, maxit=300)
print(f"max |K_T - K_fd|: correct {diff_ok:.1e}, buggy {diff_bug:.2f}")
print(f"buggy tangent: converged={ok_bug}, u = {u_bug[1:3]}, iterations = {len(norms_bug) - 1}")

# modified Newton: freeze the initial tangent
frozen = lambda d: (d + d**3, 1.0)
walls_frozen = [(0, 1, frozen), (1, 2, linear), (2, 3, linear)]
for F in (1.0, 2.5):
    fe = np.array([0.0, F, 0.0, 0.0])
    _, nn, okn = newton_solve(np.zeros(4), fe, free, walls, maxit=500)
    _, nm, okm = newton_solve(np.zeros(4), fe, free, walls_frozen, maxit=500)
    print(f"F = {F}: Newton {len(nn) - 1} iterations; initial tangent: converged={okm}, {len(nm) - 1} iterations")

In [ ]:
# ✅ check — Part 4
assert diff_ok < 1e-6, "The correct tangent must match the finite-difference derivative."
assert diff_bug > 0.5, "The check catches the bug."
assert ok_bug and np.allclose(u_bug[1:3], [1, 0.5], atol=1e-8) and len(norms_bug) - 1 > 50
print("OK. A wrong tangent does not change the answer — the residual decides that —")
print("but it destroys quadratic convergence, or convergence altogether.")

---
## Part 5 — Load steps, and a matrix of solutions

Increase the load on node 1 of the wall example from 0 to 6 in 12 equal steps. In each step start
Newton from the converged state of the previous one. Store:
* the converged free displacements of every step as a column of a matrix `S` (2 rows, 13 columns),
* the number of iterations per step.

Plot the load against $u_1$ together with the linear prediction ($u_1 = F/1.5$, from the initial
stiffness). Where do the two curves part?

In [ ]:
loads = np.linspace(0, 6, 13)
S = np.zeros((2, len(loads)))
iterations = []
u = np.zeros(4)
for k, F in enumerate(loads):
    fe = np.array([0.0, F, 0.0, 0.0])
    u, nn, ok = ...                           # TODO: start from the previous u
    S[:, k] = ...                             # TODO
    iterations.append(len(nn) - 1)
print("iterations per step:", iterations)
plt.plot(S[0], loads, 'o-', label='nonlinear (load steps)')
plt.plot(loads/1.5, loads, 'k--', label='linear prediction')
plt.xlabel('u_1'); plt.ylabel('load on node 1'); plt.legend(); plt.show()

In [ ]:
# ✅ check — Part 5
assert np.allclose(S[:, 5], [1.0, 0.5], atol=1e-8), "Step 5 is F = 2.5: the solution of Part 3."
assert max(iterations) <= 5
print("OK. Each step starts close to its answer: 3-4 iterations instead of 6.")

---
## Part 6 — A maximum load: load control fails, displacement control does not

Two springs in a row: node 0 fixed, spring 1 **softening** ($N = \Delta e^{-\Delta}$, largest force $1/e \approx 0.368$
at $\Delta = 1$), spring 2 linear; the load acts on node 2.

1. *Load control*: try $F = 0.3$ and $F = 0.45$. What happens to the second, and why can no
   solver fix it?
2. *Displacement control*: instead of the force, **prescribe** $u_2 = U$ (node 2 is then not free)
   and solve for $u_1$. The force needed is the reaction at node 2, i.e. `f_int[2]`. Sweep
   $U$ from 0 to 6 and plot force against $U$. Your `newton_solve` needs no change: prescribed
   displacements are simply non-free entries of `u` with a non-zero value.

In [ ]:
chain2 = [(0, 1, soften), (1, 2, linear)]
for F in (0.3, 0.45):
    _, nn, ok = newton_solve(np.zeros(3), np.array([0.0, 0.0, F]), [1, 2], chain2, maxit=100)
    print(f"load control, F = {F}: converged = {ok} after {len(nn) - 1} iterations")

U_values = np.linspace(0, 6, 121)
reaction = []
u = np.zeros(3)
for U in U_values:
    u[2] = U                                  # prescribed displacement
    u, nn, ok = ...                           # TODO: only node 1 is free; no external force
    reaction.append(...)                      # TODO: force needed at node 2
reaction = np.array(reaction)
plt.plot(U_values, reaction); plt.axhline(1/np.e, ls='--', c='k')
plt.xlabel('prescribed end displacement U'); plt.ylabel('force at the end'); plt.show()
print(f"largest force: {reaction.max():.4f}   (1/e = {1/np.e:.4f})")

In [ ]:
# ✅ check — Part 6
assert abs(reaction.max() - 1/np.e) < 2e-3
assert reaction[-1] < 0.05, "Far beyond the peak the softened spring carries almost nothing."
print("OK. Beyond the maximum load there is no equilibrium for a prescribed force —")
print("but there is one for every prescribed displacement. Lecture 7 (planned) builds on this.")

---
## Part 7 — Contact with a Lagrange multiplier

The two-spring chain of Lecture 6, Section 6.8: free displacements $u_2, u_3$, stiffness
$\mathbf K = \left[\begin{smallmatrix}2&-1\\-1&1\end{smallmatrix}\right]$, a load $F$ on the last node, and a rigid stop at $g = 4$.

1. Write `solve_constrained(K, f, B, g)` that builds and solves
   $\begin{bmatrix}\mathbf K & \mathbf B^T\\ \mathbf B & \mathbf 0\end{bmatrix}\begin{bmatrix}\mathbf u\\ \boldsymbol\lambda\end{bmatrix} = \begin{bmatrix}\mathbf f\\ \mathbf g\end{bmatrix}$ (use `np.block`).
2. Write `contact_solve(F)`, a tiny **active-set** loop: solve without contact; if the node passes
   the stop, solve with the constraint; if the multiplier comes out negative, release the contact.
3. Sweep $F$ from 0 to 5 and plot the contact force against $F$. At which load does contact start?

In [ ]:
Kc = np.array([[2.0, -1.0], [-1.0, 1.0]])
B = np.array([[0.0, 1.0]]); g = np.array([4.0])

def solve_constrained(K, f, B, g):
    n, c = K.shape[0], B.shape[0]
    A = ...                                   # TODO: the saddle-point matrix
    x = np.linalg.solve(A, np.concatenate([f, g]))
    return x[:n], x[n:]

def contact_solve(F):
    f = np.array([0.0, F])
    u = np.linalg.solve(Kc, f)
    lam = 0.0
    if ...:                                   # TODO: does node 3 pass the stop?
        u, l = solve_constrained(Kc, f, B, g)
        lam = l[0]
        if lam < 0:                           # a stop cannot pull: release
            # (cannot happen here; with several contacts this becomes a loop)
            u, lam = np.linalg.solve(Kc, f), 0.0
    return u, lam

for F in (3.0, 1.5):
    print(F, contact_solve(F))
Fs = np.linspace(0, 5, 51)
lams = np.array([contact_solve(F)[1] for F in Fs])
plt.plot(Fs, lams); plt.xlabel('load F'); plt.ylabel('contact force'); plt.show()

In [ ]:
# ✅ check — Part 7
u3, l3 = contact_solve(3.0)
assert np.allclose(u3, [2, 4]) and abs(l3 - 1.0) < 1e-12, "Lecture 6: u = (2, 4), lambda = 1."
assert contact_solve(1.5)[1] == 0.0
assert np.allclose(lams[Fs > 2.05], Fs[Fs > 2.05] - 2.0), "Beyond contact the stop takes every additional unit of load."
print("OK. Contact starts at F = 2; after that, the springs carry 2 and the stop carries F - 2.")

---
## Closing reflections

1. In Part 4 the buggy tangent still converged to the correct answer. Why can a wrong tangent never
   produce a wrong *converged* solution? What is it, then, that makes the answer right?
2. In Part 6, what does a commercial code report when the load exceeds the maximum under load
   control? What would you check before believing that the structure "fails" there?
3. The matrix `S` of Part 5 (and the reaction curve of Part 6) contain converged states for many
   values of a *parameter* — the load level. The snapshot matrix of Exercise 5 contained states
   for many *times*. Both are tables of solutions. If you had to store the solutions of a large
   model for a thousand load levels, would you need to store every number?
   *(Again a pointer to Lecture D on model reduction — and, for nonlinear problems, the
   reason that reduction is harder there than for linear dynamics.)*